# PulseCam – training on Colab (free GPU)
Runtime ▸ Change runtime type ▸ GPU. Checkpoints go to Google Drive so a disconnect loses nothing.
**Research prototype, not a medical device.** Never put volunteer videos in a public place.

## 1. Drive + code

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/PulseCam/runs /content/drive/MyDrive/PulseCam/data
# private repo: use a token, or upload the project folder to Drive and `%cd` into it instead
REPO = 'https://github.com/<you>/PulseCam.git'
!git clone $REPO /content/PulseCam
%cd /content/PulseCam

## 2. rPPG-Toolbox + our patch (own-recordings loader, fine-tune hook)

In [ ]:
!git clone --depth 1 https://github.com/ubicomplab/rPPG-Toolbox.git
%cd rPPG-Toolbox
!bash setup.sh uv   # if this fails on Colab: !pip install -r requirements.txt
!python ../toolbox_patch/install.py .
!ln -sfn /content/drive/MyDrive/PulseCam/runs runs   # checkpoints -> Drive
!mkdir -p ../results/logs

## 3. Data
Put datasets under `/content/drive/MyDrive/PulseCam/data/{UBFC-rPPG,PURE,SCAMPS}` (UBFC-rPPG: direct download; PURE: email the authors; SCAMPS: direct download).
Then edit `DATA_PATH` / `CACHED_PATH` in each yaml (set `DO_PREPROCESS: True` the first time). Copy yamls you edit into `../configs/train/` or `../configs/eval/`.

## 4. E1 – baselines vs deep models on UBFC-rPPG

In [ ]:
# classical GREEN / ICA / CHROM / POS (edit UNSUPERVISED.METHOD in the yaml to keep just these four)
!python main.py --config_file configs/infer_configs/UBFC-rPPG_UNSUPERVISED.yaml 2>&1 | tee ../results/logs/E1_classical.log
# pre-trained deep models on UBFC-rPPG
for m in ['PHYSNET','TSCAN','DEEPPHYS']:
    !python main.py --config_file configs/infer_configs/PURE_UBFC-rPPG_{m}_BASIC.yaml 2>&1 | tee ../results/logs/E1_pretrained_{m}.log

Also train your own on UBFC subjects (copy a train config, set TRAIN/VALID/TEST DATASET to UBFC-rPPG with BEGIN/END subject splits) and `tee` to `results/logs/E1_trained_<model>.log`.

## 5. E2 – cross-dataset (train PURE or SCAMPS, test UBFC-rPPG)

In [ ]:
!python main.py --config_file configs/train_configs/PURE_PURE_UBFC-rPPG_TSCAN_BASIC.yaml 2>&1 | tee ../results/logs/E2_PURE_to_UBFC_tscan.log
# if PURE access is late:
# !python main.py --config_file configs/train_configs/SCAMPS_SCAMPS_UBFC-rPPG_TSCAN_BASIC.yaml 2>&1 | tee ../results/logs/E2_SCAMPS_to_UBFC_tscan.log

## 6. Fine-tune on Indian volunteers (E4)
Copy `data/own/videos` to Colab (private Drive), fix the paths in the yaml, then:

In [ ]:
import os; os.environ['PULSECAM_INIT'] = 'final_model_release/PURE_TSCAN.pth'
!python main.py --config_file ../configs/train/PulseCam_finetune_TSCAN.yaml 2>&1 | tee ../results/logs/E4_finetune.log

## 7. Collect results + save the best weights

In [ ]:
!python ../eval/parse_logs.py
import glob, shutil
ck = sorted(glob.glob('runs/exp/*/PreTrainedModels/*Epoch*.pth') or glob.glob('runs/**/*Epoch*.pth', recursive=True))
print(ck[-3:])
# shutil.copy(ck[-1], '../models/best_tscan.pth')  # pick the checkpoint that beat the baselines; the demo then uses it